# Stage 3 — With GAN, No Drift
### GPVS-Faults | Journal extension of ICSPCS 2024 & ITNAC 2026

Stage 3 = Stage 1's undrifted data + per-class WGAN-GP training-set
augmentation (`wgans.py`), then the same two classifiers re-evaluated.
This is the roadmap's **Cell 2**: "pure augmentation benefit (replicates
ITNAC)" — comparing this stage against Stage 1 isolates whether GAN
augmentation helps at all, before drift enters in Stage 4.

All model-training code is unchanged from Stage 1/2. The only new
machinery is `gan_augmentation.py`, which wraps `wgans.py`'s
`train_class_gans`/`augment` API and handles conversion between this
project's `splits` representation and the flat array wgans.py expects.

**Design decisions** (full rationale in `gan_augmentation.py`'s docstring):

1. **Raw-space augmentation, frozen scaler last.** GAN training and
   generation happen in raw (unscaled) feature space — the same pipeline
   slot Phase 3/4 drift injection occupied in Stage 2. The frozen scaler
   (fit once on Stage 1's raw Normal training data, unchanged since Stage 1)
   is applied last, identically to real and synthetic rows.
2. **Train-only augmentation.** val/test are untouched — same fixed-evaluation-set
   discipline as every other stage.
3. **One fixed GAN-training seed** (`GAN_SEED`), independent of the 5
   classifier-training seeds swept below. Isolates classifier-seed variance
   (already characterized in Stages 1-2) from GAN-training variance (a
   separate, unexplored axis — flagged as a follow-up, not a blocker).
4. **Augmentation ratio.** `GAN_AUGMENT_RATIO = 1.0` (one synthetic row per
   real row per class) is used as a default here since ITNAC's own exact
   ratio wasn't available at build time. *If ITNAC used a specific ratio,
   set `GAN_AUGMENT_RATIO` to match it for a true "replicates ITNAC"
   comparison.*
5. **No single-seed anchor this time.** Stages 1-2 showed a single seed can
   be badly misleading (the seed-0 LSTM-XGB comparison overstated Stage 2's
   drift effect by ~9 points). This notebook goes straight to the 5-seed
   evaluation.

> **Execution note:** this sandbox has no `torch`/GPU, so every cell that
> trains or calls the WGAN-GP or the classifiers is included as ready-to-run
> code but was not executed here. `gan_augmentation.py`'s conversion/rebuild
> logic (splits ⇄ flat array, val/test preservation, real/synthetic row
> ordering) *was* unit-tested against the real `base_splits.pkl` outside
> this notebook and is correct — only the GPU-bound steps need your
> environment.


In [1]:
import pandas as pd, numpy as np, os, sys, torch, torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
sys.path.insert(0, "..")
from base_splits import load_splits, summarize_splits
from gan_augmentation import gan_augment_splits, splits_train_to_array
from multiseed import run_multi_seed, aggregate_results, summary_table, per_class_accuracy, paired_diff, DEFAULT_SEEDS
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from scipy.stats import ks_2samp, wasserstein_distance
from models.lstm_xgb import LSTM_XGB
from models.cnn_lstm import CNN_LSTM_v1, CNN_LSTM_v2
from utils import reset_gpu_peak_memory, get_gpu_peak_memory_mb, measure_inference_time, Timer, count_parameters

splits_raw = load_splits(path="../base_splits.pkl")
print(summarize_splits(splits_raw))

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

cols = list(splits_raw[0].train.columns[1:-1])
print("feature cols:", cols)


Loaded base splits for 8 classes from /home/maddie/cd-study/Stages/base_splits.pkl
   label  train_n  val_n  test_n  time_start   time_end
0      0     1000    300     300    3.993639   4.153525
1      1     1000    300     300    9.156422   9.316307
2      2     1000    300     300    4.507887   4.667771
3      3     1000    300     300    4.866492   5.026376
4      4     1000    300     300    1.528859   1.688743
5      5     1000    300     300    8.582310   8.742194
6      6     1000    300     300    8.765543   8.925427
7      7     1000    300     300   10.680525  10.840409


device: NVIDIA A100-PCIE-40GB
feature cols: ['Ipv', 'Vpv', 'Vdc', 'ia', 'ib', 'ic', 'va', 'vb', 'vc', 'Iabc', 'If', 'Vabc', 'Vf']


## GAN Training & Augmentation

One WGAN-GP per class, trained on that class's real raw training rows only,
then `GAN_AUGMENT_RATIO` synthetic rows per real row are generated and
appended to the training set. `gan_augment_splits` (in `gan_augmentation.py`)
handles the whole round trip.


In [2]:
GAN_SEED = 20260827          # fixed GAN-training seed, independent of classifier seeds
GAN_AUGMENT_RATIO = 1.0      # synthetic:real ratio per class -- see design note #4 above

splits_stage3, gans, gan_histories, n_real = gan_augment_splits(
    splits_raw, seed=GAN_SEED, device=device, ratio=GAN_AUGMENT_RATIO)

print(f"Trained {len(gans)} per-class WGAN-GP generators.")
for lbl, hist in sorted(gan_histories.items()):
    print(f"  class {lbl}: final W_dist={hist['w_dist'][-1]:.4f}  "
          f"(n_real={gans[lbl][2]})")

for lbl in sorted(splits_stage3):
    n_train = len(splits_stage3[lbl].train)
    print(f"  class {lbl}: train={n_train} (real={n_real // 8}, synthetic={n_train - n_real // 8})")


/home/maddie/miniconda3/envs/cd/lib/python3.10/site-packages/torch/autograd/graph.py:769: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Trained 8 per-class WGAN-GP generators.
  class 0: final W_dist=0.3082  (n_real=1000)
  class 1: final W_dist=0.2718  (n_real=1000)
  class 2: final W_dist=0.2258  (n_real=1000)
  class 3: final W_dist=0.2180  (n_real=1000)
  class 4: final W_dist=0.2735  (n_real=1000)
  class 5: final W_dist=0.3258  (n_real=1000)
  class 6: final W_dist=0.3373  (n_real=1000)
  class 7: final W_dist=0.2556  (n_real=1000)
  class 0: train=2000 (real=1000, synthetic=1000)
  class 1: train=2000 (real=1000, synthetic=1000)
  class 2: train=2000 (real=1000, synthetic=1000)
  class 3: train=2000 (real=1000, synthetic=1000)
  class 4: train=2000 (real=1000, synthetic=1000)
  class 5: train=2000 (real=1000, synthetic=1000)
  class 6: train=2000 (real=1000, synthetic=1000)
  class 7: train=2000 (real=1000, synthetic=1000)


## GAN Fidelity Check

Before trusting any accuracy change to augmentation, confirm the synthetic
rows actually resemble the real training distribution they're supposed to
extend — the same KS/Wasserstein manipulation-check style used for the
drift injection in Stage 2, just checking similarity instead of engineered
dissimilarity.


In [3]:
fidelity_rows = []
for lbl in sorted(splits_stage3):
    real = splits_raw[lbl].train
    aug = splits_stage3[lbl].train
    synth = aug.iloc[len(real):]   # rows after the real ones are synthetic (see gan_augment_splits docstring)
    for feat in cols:
        ks = ks_2samp(real[feat], synth[feat]).statistic
        w = wasserstein_distance(real[feat], synth[feat])
        fidelity_rows.append({"class": lbl, "feature": feat, "ks": ks, "wasserstein": w})

fidelity = pd.DataFrame(fidelity_rows)
print("GAN fidelity (real train vs. synthetic), mean across classes per feature:")
print(fidelity.pivot_table(index="feature", values=["ks", "wasserstein"], aggfunc="mean").round(4))
print("\nWorst-fidelity (class, feature) pairs by KS -- worth a visual check if any stand out:")
print(fidelity.nlargest(10, "ks").to_string(index=False))


GAN fidelity (real train vs. synthetic), mean across classes per feature:
             ks  wasserstein
feature                     
Iabc     0.0976       0.0003
If       0.0766       0.0083
Ipv      0.1498       0.0171
Vabc     0.0882       0.0051
Vdc      0.3020       0.1860
Vf       0.0764       0.0006
Vpv      0.2539       0.1332
ia       0.0540       0.0484
ib       0.0506       0.0481
ic       0.0596       0.0472
va       0.0549       7.0516
vb       0.0474       5.0014
vc       0.0598       7.3592

Worst-fidelity (class, feature) pairs by KS -- worth a visual check if any stand out:
 class feature    ks  wasserstein
     4     Vpv 0.778     0.333498
     5     Vdc 0.757     0.295251
     2     Vpv 0.482     0.297054
     6     Vdc 0.349     0.311775
     0     Vdc 0.266     0.183529
     5     Vpv 0.257     0.038059
     2     Vdc 0.254     0.142337
     5     Ipv 0.248     0.006058
     0     Vpv 0.240     0.147980
     0     Ipv 0.231     0.037641


## Frozen Scaler + Z-Scaling

Same frozen scaler as every stage: fit once on Stage 1's raw, undrifted
Normal-class training data, reused via `.transform()` only.


In [4]:
scaler = StandardScaler()
scaler.fit(splits_raw[0].train[cols])   # frozen -- identical fit to Stage 1/2, never refit

dct = dict()
for i in range(len(splits_stage3)):
    dct[i] = dict()
    dct[i].update({
        "train": pd.DataFrame(scaler.transform(splits_stage3[i].train[cols]), columns=cols,
                              index=splits_stage3[i].train.index).assign(Fault=i),
        "val": pd.DataFrame(scaler.transform(splits_stage3[i].val[cols]), columns=cols,
                            index=splits_stage3[i].val.index).assign(Fault=i),
        "test": pd.DataFrame(scaler.transform(splits_stage3[i].test[cols]), columns=cols,
                             index=splits_stage3[i].test.index).assign(Fault=i),
    })
print("Stage 3 dct built:", {i: {k: len(v) for k, v in dct[i].items()} for i in dct})


Stage 3 dct built: {0: {'train': 2000, 'val': 300, 'test': 300}, 1: {'train': 2000, 'val': 300, 'test': 300}, 2: {'train': 2000, 'val': 300, 'test': 300}, 3: {'train': 2000, 'val': 300, 'test': 300}, 4: {'train': 2000, 'val': 300, 'test': 300}, 5: {'train': 2000, 'val': 300, 'test': 300}, 6: {'train': 2000, 'val': 300, 'test': 300}, 7: {'train': 2000, 'val': 300, 'test': 300}}


## Model Evaluation — LSTM-XGB and CNN-LSTM-v2 on Stage 3 Data

Unchanged from Stage 1/2: same `to_tensors`, `load_scenario`, `run_scenario`,
CNN-LSTM training loop, only pointed at the Stage 3 `dct`.


In [5]:
def to_tensors(df, cols):
    """Convert a (features + Fault) DataFrame into model-ready tensors.
    X: (N, 1, len(cols)) so the LSTM sees the len(cols) features as a
       length-len(cols) sequence with 1 channel each (matches LSTM_XGB's
       expected input shape).
    y: (N,) integer Fault labels.
    """
    X = torch.from_numpy(df[cols].to_numpy(dtype="float32")).unsqueeze(1)
    y = torch.from_numpy(df["Fault"].to_numpy(dtype="int64"))
    return X, y


def load_scenario(dct, cols):
    """Build combined 8-class train/val/test tensors directly from the
    in-memory `dct` dict (built in the Z-scale cell), instead of reading
    per-scenario CSVs off disk.

    dct is keyed by class label: dct[i]["train"/"val"/"test"] is a
    per-class DataFrame of z-scored features + a Fault column. We
    concatenate across classes to get the full multiclass split.
    """
    train_df = pd.concat([dct[i]["train"] for i in sorted(dct)], axis=0)
    val_df   = pd.concat([dct[i]["val"]   for i in sorted(dct)], axis=0)
    test_df  = pd.concat([dct[i]["test"]  for i in sorted(dct)], axis=0)
    return (to_tensors(train_df, cols),
            to_tensors(val_df,   cols),
            to_tensors(test_df,  cols))


def run_scenario(scenario_idx, dct, cols, device,
                 epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(dct, cols)

    model = LSTM_XGB(n_classes=8, lstm_hidden=32, device=device, seed=seed)
    n_params, params_by_type = count_parameters(model.backbone)  # LSTM only
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} (LSTM_XGB) ===")
        print(f"  LSTM params: {n_params:,}  breakdown: {params_by_type}")

    # ---- Stage 1: LSTM training ----
    with Timer(device) as stage1_timer:
        model.fit_lstm(X_tr, y_tr, X_val=X_va, y_val=y_va,
                       epochs=epochs, lr=lr, weight_decay=weight_decay,
                       batch_size=50, seed=seed, verbose=verbose)

    # ---- Stage 2: XGBoost fitting ----
    with Timer(device=None) as stage2_timer:   # XGBoost is CPU-bound
        model.fit_xgb(X_tr, y_tr)

    train_sec_total = stage1_timer.elapsed + stage2_timer.elapsed
    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- Inference timing ----
    X_te_dev = X_te.to(device)
    inf_stats = measure_inference_time(model.predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- Test accuracy ----
    y_true = y_te.numpy()
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: stage1(LSTM)={stage1_timer.elapsed:.1f}s  "
              f"stage2(XGB)={stage2_timer.elapsed:.1f}s  "
              f"total={train_sec_total:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample")

    return {
        "scenario": scenario_idx, "model": "LSTM_XGB",
        "n_train": len(X_tr),
        "accuracy": acc, "precision": p, "recall": r, "f1": f,
        "confusion": cm,
        "n_params": n_params,
        "train_sec": round(train_sec_total, 2),
        "train_sec_stage1": round(stage1_timer.elapsed, 2),
        "train_sec_stage2": round(stage2_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }


## Multi-Seed Evaluation (LSTM-XGB)

In [6]:
print(f"LSTM-XGB across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
lstm_xgb_seed_results = run_multi_seed(
    run_scenario, seeds=DEFAULT_SEEDS, scenario_idx=0, dct=dct, cols=cols, device=device)

lstm_xgb_agg = aggregate_results(lstm_xgb_seed_results)
print("\nLSTM-XGB, mean +/- std across seeds:")
for m, s in lstm_xgb_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}   (min={s['min']:.4f} max={s['max']:.4f})")


LSTM-XGB across 5 seeds: [0, 1, 2, 3, 4]


  seed=0  acc=0.9608  P=0.9698  R=0.9608  F1=0.9600  train_sec=87.8


  seed=1  acc=0.9908  P=0.9914  R=0.9908  F1=0.9908  train_sec=87.3


  seed=2  acc=0.9113  P=0.9205  R=0.9113  F1=0.9084  train_sec=87.1


  seed=3  acc=0.9967  P=0.9967  R=0.9967  F1=0.9967  train_sec=88.6


  seed=4  acc=0.9050  P=0.9335  R=0.9050  F1=0.8912  train_sec=87.8

LSTM-XGB, mean +/- std across seeds:
  accuracy    0.9529 +/- 0.0431   (min=0.9050 max=0.9967)
  precision   0.9624 +/- 0.0341   (min=0.9205 max=0.9967)
  recall      0.9529 +/- 0.0431   (min=0.9050 max=0.9967)
  f1          0.9494 +/- 0.0478   (min=0.8912 max=0.9967)


In [7]:
# CNN-LSTM training pipeline -- byte-identical to stage1.ipynb. to_tensors()
# and load_scenario() are reused as-is from the previous cell.

def make_model(model_cls, device, seed=0, **kwargs):
    """Fresh CNN-LSTM with Xavier init for Conv/Linear; default PyTorch init
    for LSTM (MATLAB's Glorot applies to Conv and FC, LSTM uses its own)."""
    torch.manual_seed(seed)
    model = model_cls(n_classes=8, **kwargs).to(device)
    for m in model.modules():
        if isinstance(m, nn.Conv1d):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=0):
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(X_tr), generator=g)
    X_tr_s, y_tr_s = X_tr[perm], y_tr[perm]
    train_loader = DataLoader(TensorDataset(X_tr_s, y_tr_s),
                              batch_size=batch_size, shuffle=False)
    val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size)
    return train_loader, val_loader, test_loader


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate_loader(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss    += criterion(logits, yb).item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for xb, yb in loader:
        logits = model(xb.to(device))
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(yb.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)


def run_scenario_cnn_lstm(scenario_idx, dct, cols, model_cls, device,
                          epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(dct, cols)
    train_loader, val_loader, test_loader = make_loaders(
        X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=seed)

    model = make_model(model_cls, device, seed=seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr,
                           betas=(0.9, 0.999), eps=1e-8,
                           weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    n_params, params_by_type = count_parameters(model)
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} ({model_cls.__name__}) ===")
        print(f"  params: {n_params:,}  breakdown: {params_by_type}")

    with Timer(device) as train_timer:
        for ep in range(1, epochs + 1):
            tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                              criterion, optimizer, device)
            va_loss, va_acc = evaluate_loader(model, val_loader,
                                              criterion, device)
            scheduler.step()
            if verbose and (ep == 1 or ep % 10 == 0 or ep == epochs):
                print(f"  ep {ep:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f}"
                      f" | val loss {va_loss:.4f} acc {va_acc:.3f}")

    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- Inference timing ----
    X_te_dev = X_te.to(device)
    def _predict(X):
        model.eval()
        with torch.no_grad():
            return model(X).argmax(1)
    inf_stats = measure_inference_time(_predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- Test accuracy ----
    y_true, y_pred = predict_all(model, test_loader, device)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: train={train_timer.elapsed:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample  "
              f"peak_mem={peak_mem_mb:.1f}MB")

    return {
        "scenario":    scenario_idx,
        "model":       model_cls.__name__,
        "n_train":     len(X_tr),
        "accuracy":    acc,
        "precision":   p,
        "recall":      r,
        "f1":          f,
        "confusion":   cm,
        "n_params":    n_params,
        "train_sec":   round(train_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }


## Multi-Seed Evaluation (CNN-LSTM-v2)

In [8]:
MODEL_CLS = CNN_LSTM_v2  # or CNN_LSTM_v1

print(f"CNN-LSTM-v2 across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
cnn_lstm_seed_results = run_multi_seed(
    run_scenario_cnn_lstm, seeds=DEFAULT_SEEDS, scenario_idx=1, dct=dct, cols=cols,
    model_cls=MODEL_CLS, device=device)

cnn_lstm_agg = aggregate_results(cnn_lstm_seed_results)
print("\nCNN-LSTM-v2, mean +/- std across seeds:")
for m, s in cnn_lstm_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}   (min={s['min']:.4f} max={s['max']:.4f})")


CNN-LSTM-v2 across 5 seeds: [0, 1, 2, 3, 4]


  seed=0  acc=0.9996  P=0.9996  R=0.9996  F1=0.9996  train_sec=135.7


  seed=1  acc=1.0000  P=1.0000  R=1.0000  F1=1.0000  train_sec=135.0


  seed=2  acc=1.0000  P=1.0000  R=1.0000  F1=1.0000  train_sec=126.6


  seed=3  acc=0.9996  P=0.9996  R=0.9996  F1=0.9996  train_sec=134.0


  seed=4  acc=0.9996  P=0.9996  R=0.9996  F1=0.9996  train_sec=134.5

CNN-LSTM-v2, mean +/- std across seeds:
  accuracy    0.9998 +/- 0.0002   (min=0.9996 max=1.0000)
  precision   0.9998 +/- 0.0002   (min=0.9996 max=1.0000)
  recall      0.9997 +/- 0.0002   (min=0.9996 max=1.0000)
  f1          0.9997 +/- 0.0002   (min=0.9996 max=1.0000)


## Combined Multi-Seed Summary

In [9]:
combined = summary_table({"LSTM-XGB": lstm_xgb_agg, "CNN-LSTM-v2": cnn_lstm_agg})
print(combined.round(4).to_string(index=False))

class_names = ["Normal", "F1", "F2", "F3", "F4", "F5", "F6", "F7"]
pc = pd.DataFrame({
    "LSTM-XGB": per_class_accuracy(lstm_xgb_agg["mean_confusion_rate"], class_names),
    "CNN-LSTM-v2": per_class_accuracy(cnn_lstm_agg["mean_confusion_rate"], class_names),
})
print("\nPer-class accuracy (mean confusion diagonal across seeds):")
print(pc.round(4).to_string())

import pickle
with open("stage3_seed_results.pkl", "wb") as f:
    pickle.dump({"lstm_xgb": lstm_xgb_agg, "cnn_lstm_v2": cnn_lstm_agg}, f)
print("\nSaved stage3_seed_results.pkl")


      model  n_seeds  accuracy_mean  accuracy_std  precision_mean  precision_std  recall_mean  recall_std  f1_mean  f1_std
   LSTM-XGB        5         0.9529        0.0431          0.9624         0.0341       0.9529      0.0431   0.9494  0.0478
CNN-LSTM-v2        5         0.9998        0.0002          0.9998         0.0002       0.9998      0.0002   0.9997  0.0002

Per-class accuracy (mean confusion diagonal across seeds):
        LSTM-XGB  CNN-LSTM-v2
Normal    0.8493        1.000
F1        1.0000        1.000
F2        1.0000        1.000
F3        0.8280        0.998
F4        1.0000        1.000
F5        1.0000        1.000
F6        0.9460        1.000
F7        1.0000        1.000

Saved stage3_seed_results.pkl


## Cell 1 vs. Cell 2 — Pure GAN Benefit (No Drift)

Paired, seed-matched comparison against Stage 1's saved results. This is
the roadmap's "Cell 1 vs Cell 2" row: does augmentation help at all, with
no drift in the picture yet.


In [10]:
import pickle
with open("../stage1/stage1_seed_results.pkl", "rb") as f:
    stage1 = pickle.load(f)

for model_key, model_label in [("lstm_xgb", "LSTM-XGB"), ("cnn_lstm_v2", "CNN-LSTM-v2")]:
    stage1_results = stage1[model_key]["raw_results"]
    stage3_results = (lstm_xgb_agg if model_key == "lstm_xgb" else cnn_lstm_agg)["raw_results"]
    diff_df, diff_stats = paired_diff(stage3_results, stage1_results, metric="accuracy")
    print(f"\n{model_label}  Stage 3 (GAN) vs. Stage 1 (no GAN), paired by seed:")
    print(diff_df.round(4).to_string(index=False))
    print(f"  mean diff = {diff_stats['mean_diff']:+.4f}   "
          f"t = {diff_stats['t_stat']:.3f}   p = {diff_stats['p_value']:.4f}")



LSTM-XGB  Stage 3 (GAN) vs. Stage 1 (no GAN), paired by seed:
 seed      a      b    diff
    0 0.9608 0.9938 -0.0329
    1 0.9908 0.8850  0.1058
    2 0.9112 0.9883 -0.0771
    3 0.9967 0.9962  0.0004
    4 0.9050 0.9888 -0.0837
  mean diff = -0.0175   t = -0.508   p = 0.6380

CNN-LSTM-v2  Stage 3 (GAN) vs. Stage 1 (no GAN), paired by seed:
 seed      a      b    diff
    0 0.9996 1.0000 -0.0004
    1 1.0000 1.0000  0.0000
    2 1.0000 1.0000  0.0000
    3 0.9996 1.0000 -0.0004
    4 0.9996 0.9996  0.0000
  mean diff = -0.0002   t = -1.633   p = 0.1778


## Next Steps

- Run this notebook end-to-end in your GPU environment. Check the GAN
  fidelity table first — if any class/feature pair shows high KS between
  real and synthetic, inspect that class's WGAN-GP training curve
  (`gan_histories[lbl]["w_dist"]`) before trusting its augmented results.
- Compare Stage 3 vs. Stage 1 per-class accuracy specifically for F6/F7
  (Stage 1's known borderline classes) and for the classes that were most
  seed-unstable in Stage 1 (Normal, F1, F6) — augmentation's clearest
  possible benefit is stabilizing exactly those.
- Stage 4 reuses `gan_augment_splits` and `drift_injection.py` unchanged,
  applying GAN augmentation on top of Stage 2's already-drifted training
  data, plus an optional drift-aware "domain adaptation" variant (Stage 4b).
